In [1]:
pip install langchain faiss-cpu sentence-transformers huggingface-hub transformers langchain-community langchain-huggingface torch

Note: you may need to restart the kernel to use updated packages.


In [1]:
from langchain_core._api.deprecation import LangChainDeprecationWarning
import warnings

warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

In [4]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [2]:

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"C:\Users\VISHAL KIRTHIK\Downloads\cons_law.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages")

C:\Users\VISHAL KIRTHIK\AppData\Local\Temp\ipykernel_20580\1068572291.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 402 pages


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# chunk_size raised 800 -> 1500: Indian Constitution articles/clauses regularly
# run past 800 chars, so the old size was slicing single provisions in half.
# chunk_overlap cut 500 -> 200: 500/800 was a ~62% overlap, which floods the
# index with near-duplicate chunks and doesn't help retrieval quality.
# 200/1500 (~13%) is enough to avoid losing context at a chunk boundary.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(docs)
print(f"Number of chunks: {len(chunks)}")
print(chunks[1].page_content)

Number of chunks: 764
PREFACE 
 
This is the  sixth pocket size edition of the Constitution of 
India in the diglot form. In this edition, the text of the Constitution 
of India has been brought up-to-date by incorporating therein all 
the amendments up to the Constitution (One Hundred and Sixth 
Amendment) Act, 2023. The foot notes below the text indicate the 
Constitution Amendment Acts by which such amendments have 
been made. 
The Constitution (One Hundredth Amendment) Act, 2015 
containing details of acquired and transferred territories between 
the Governments of India and Bangladesh has been provided in 
Appendix I. 
The Constitution (Application to Jammu and Kashmir) 
Order, 2019 and the declaration under article 370(3) of the 
Constitution have been provided respe ctively in Appendix II and 
Appendix III for reference. 
 
 
New Delhi; Dr. Rajiv Mani, 
1st May, 2026 Secretary to the Government of India.


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [31]:
# Optional sanity check — confirms the embedding model works
sample_chunk = "Artificial Intelligence enables machines to perform tasks that typically require human intelligence"
vector = embeddings.embed_query(sample_chunk)
len(vector)

384

In [5]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

db = FAISS.from_documents(
    chunks,
    embeddings,
    distance_strategy=DistanceStrategy.COSINE,
    normalize_L2=True,
)

C:\Anaconda\Lib\site-packages\langchain_community\vectorstores\faiss.py:233: UserWarning: Normalizing L2 is not applicable for metric type: DistanceStrategy.COSINE
  warnings.warn(


In [ ]:
# Plain top-k similarity retrieval -- no hard score-threshold gate.
# A fixed similarity cutoff was rejecting valid, answerable questions
# (e.g. "what is a High Court?") because a broad/general question doesn't
# score as high on embedding similarity as a question that echoes the
# document's exact wording, even when a directly relevant Article exists.

In [6]:
# Plain top-k similarity search. k=4 gives the model a bit more context to
# work with for broader/definitional questions ("what is a High Court")
# that don't lexically match one narrow chunk as tightly as a question that
# quotes the Constitution's own phrasing.
retriever = db.as_retriever(search_kwargs={"k": 4})

In [7]:
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device=-1,                      # CPU
    # Removed torch_dtype=torch.bfloat16. This was the actual cause of the
    # garbled, letters-fused-together output: most CPUs have no native bf16
    # matmul support, so PyTorch emulates/rounds it, and on a small 1.5B model
    # that shows up as corrupted generations. float32 (the default when you
    # omit torch_dtype) is slower but numerically correct on CPU.
    max_new_tokens=120,
    temperature=0.2,
    do_sample=False,
    repetition_penalty=1.15,        # 1.5 was aggressive enough to distort output
)
pipe.tokenizer.pad_token_id = pipe.tokenizer.eos_token_id

from langchain_huggingface import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [36]:
PROMPT_BASIC_GROUNDED_RAG = """
You are an AI assistant answering questions using retrieved documents.

Instructions:
- Use only the provided context.
- Do not use outside knowledge.
- If the context is insufficient, explicitly state that you do not have enough information.
- Do not invent facts.
- Prefer information appearing in multiple retrieved documents.
- Keep the response concise.
- If multiple answers exist in the context, mention them.
"""

PROMPT_REVISION_V1 = """
Revise the answer.

Requirements:
- Remove hallucinations.
- Fill missing supported information.
- Resolve contradictions.
- Preserve original meaning.
- Keep under 150 words.
"""



In [8]:
import re
import logging

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("self_rag")

ANSWER_FORMAT_RE = re.compile(
    r"Section\s*:\s*.+\n\s*Answer\s*:\s*.+", re.IGNORECASE
)


def ask_llm(system_msg: str, user_msg: str) -> str:
    """Send a chat-formatted prompt to the LLM and return only the new assistant reply."""
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
    ]
    output = pipe(messages)
    return output[0]["generated_text"][-1]["content"].strip()


def extract_score(label: str, text: str, default: int = 3) -> int:
    """Pull the first integer following a label, tolerant of extra text/formatting.
    Logs a warning (instead of silently guessing) when the label can't be found,
    so a default score doesn't get mistaken for a real one."""
    match = re.search(rf"{label}\s*:\s*(\d+)", text, re.IGNORECASE)
    if not match:
        logger.warning(
            "extract_score: could not find '%s' in critique output -- "
            "defaulting to %d. Raw critique text: %r",
            label, default, text,
        )
        return default
    return int(match.group(1))


def generate_answer(question: str, context: str) -> str:
    system_msg = (
        "You are a Constitutional Law citation assistant for the Constitution "
        "of India. Answer using the provided context retrieved from the "
        "Constitution of India. The question may be a general/definitional "
        "question (e.g. 'what is a High Court') -- if the context describes "
        "the relevant institution, power, or provision, answer from it even "
        "if the question doesn't quote the Constitution's exact wording.\n\n"
        "Rules:\n"
        "1. Use only the provided context. Do not use outside knowledge.\n"
        "2. Do not invent Articles, clauses, or legal interpretations.\n"
        "3. If the context genuinely does not cover the question, reply exactly:\n"
        "'The provided Constitution of India documents do not contain sufficient information to answer this question.'\n"
        "4. Output format is EXACTLY two lines and nothing else:\n"
        "Section: <Article/clause number as it appears in the context>\n"
        "Answer: <one or two short, plain sentences -- no preamble, no elaboration>\n"
        "5. Do not provide legal advice. Do not speculate."
    )

    user_msg = f"""Context:
{context}

Question: {question}

Respond in the exact two-line 'Section:' / 'Answer:' format described above. Nothing else."""

    return ask_llm(system_msg, user_msg)


def enforce_answer_format(question: str, context: str, answer: str) -> str:
    """Make sure the model actually produced the 'Section: / Answer:' format.
    Small models drift from instructed formats fairly often. If the first
    attempt doesn't match, ask once more with a stricter, narrower prompt.
    If it still doesn't match, wrap whatever text we have in the expected
    shape rather than silently passing malformed output to the user."""
    if ANSWER_FORMAT_RE.search(answer):
        return answer

    logger.warning(
        "generate_answer: output did not match 'Section:'/'Answer:' format, "
        "retrying once. Raw output: %r", answer,
    )

    retry_system_msg = (
        "Reformat the text below into EXACTLY two lines and nothing else:\n"
        "Section: <Article/clause number>\n"
        "Answer: <one or two short sentences>\n"
        "Do not add commentary. Do not add a preamble. Only output the two lines."
    )
    retry_user_msg = f"Text to reformat:\n{answer}"
    retried = ask_llm(retry_system_msg, retry_user_msg)

    if ANSWER_FORMAT_RE.search(retried):
        return retried

    logger.warning(
        "generate_answer: retry also failed to match format. "
        "Falling back to a templated wrapper. Raw retry output: %r", retried,
    )
    return f"Section: Unspecified\nAnswer: {answer.strip()}"


def critique_answer(question: str, context: str, answer: str) -> dict:
    """Score the answer on faithfulness/completeness/relevance/safety. Returns a dict of ints."""
    system_msg = (
        "You are a strict evaluator of legal RAG answers. You output ONLY "
        "numeric scores in the exact format requested -- no explanations, "
        "no extra commentary, no repeated questions."
    )
    user_msg = f"""Context:
{context}

Question: {question}

Answer:
{answer}

Score the answer from 0 to 5 on each criterion below. Respond in EXACTLY this format, nothing else:

Faithfulness: <number>
Completeness: <number>
Relevance: <number>
Safety: <number>"""

    critique_text = ask_llm(system_msg, user_msg)

    return {
        "faithfulness": extract_score("Faithfulness", critique_text),
        "completeness": extract_score("Completeness", critique_text),
        "relevance": extract_score("Relevance", critique_text),
        "safety": extract_score("Safety", critique_text),
        "raw": critique_text,   # keep raw text around for debugging
    }


def revise_answer(question: str, context: str, answer: str) -> str:
    system_msg = (
        "You are revising a draft legal citation answer. Correct any "
        "inaccuracies, remove anything not supported by the context, and "
        "keep the EXACT two-line format:\n"
        "Section: <Article/clause number>\n"
        "Answer: <one or two short sentences>\n"
        "Do not add new information beyond the context. Output only the "
        "corrected two lines -- no preamble, no explanation."
    )
    user_msg = f"""Context:
{context}

Question: {question}

Draft answer:
{answer}

Provide the corrected answer in the exact 'Section:' / 'Answer:' format."""

    return ask_llm(system_msg, user_msg)


INSUFFICIENT_CONTEXT_MSG = (
    "The provided Constitution of India documents do not contain "
    "sufficient information to answer this question."
)

GENERIC_ERROR_MSG = (
    "Something went wrong while answering this question. Please try again."
)


def retrieve_with_retry(question: str, verbose: bool = False):
    """Retrieve context, and if nothing comes back, actually retry with a
    wider k instead of just labeling the failure 'RETRIEVE AGAIN' and giving
    up. Returns the doc list (possibly empty if both attempts fail)."""
    docs = retriever.invoke(question)
    if docs:
        return docs

    if verbose:
        print("--- No results on first retrieval, retrying with wider k ---")

    try:
        wider_retriever = db.as_retriever(search_kwargs={"k": 8})
        docs = wider_retriever.invoke(question)
    except Exception:
        logger.exception("retrieve_with_retry: widened retrieval attempt failed")
        docs = []

    return docs


def self_rag_part(question: str, verbose: bool = False) -> str:
    # 1. Retrieve, with one real retry (wider k) before giving up.
    try:
        docs = retrieve_with_retry(question, verbose=verbose)
    except Exception:
        logger.exception("self_rag_part: retrieval failed")
        return GENERIC_ERROR_MSG

    if not docs:
        return INSUFFICIENT_CONTEXT_MSG

    context = "\n\n".join(doc.page_content for doc in docs)

    # 2. Generate answer, then enforce the Section/Answer format
    try:
        answer = generate_answer(question, context)
        answer = enforce_answer_format(question, context, answer)
    except Exception:
        logger.exception("self_rag_part: answer generation failed")
        return GENERIC_ERROR_MSG

    # 3. Critique
    try:
        scores = critique_answer(question, context, answer)
    except Exception:
        logger.exception("self_rag_part: critique failed -- returning ungraded answer")
        return answer

    if verbose:
        print("--- Critique scores ---")
        print(scores)

    # 4. Decision, made in Python (not by the LLM)
    if (
        scores["faithfulness"] >= 4
        and scores["completeness"] >= 4
        and scores["relevance"] >= 4
        and scores["safety"] >= 4
    ):
        decision = "ACCEPT"
    elif scores["relevance"] <= 2 or scores["completeness"] <= 2:
        decision = "INSUFFICIENT"
    else:
        decision = "REVISE"

    if verbose:
        print(f"--- Decision: {decision} ---")

    # 5. Act on decision
    if decision == "ACCEPT":
        return answer
    elif decision == "REVISE":
        try:
            revised = revise_answer(question, context, answer)
            return enforce_answer_format(question, context, revised)
        except Exception:
            logger.exception("self_rag_part: revision failed -- returning original answer")
            return answer
    else:
        return (
            "The retrieved context does not sufficiently answer this "
            "question. Please rephrase or ask about a more specific "
            "provision of the Constitution."
        )

In [19]:
question = "What is sla?"
result = self_rag_part(question)
print(result)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The retrieved context does not sufficiently answer this question. Please rephrase or ask about a more specific provision of the Constitution.


In [ ]:
What are the Fundamental Duties?
How many Fundamental Duties are there?
Is protecting the environment a Fundamental Duty?
Which Article deals with Fundamental Duties